In [1]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\test'

Test du fichier prediction.py

In [2]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

In [4]:
import joblib 
import pandas as pd
from pathlib import Path
from mlProject.constants import *
from mlProject.utils.common import read_yaml

In [5]:
#  input_data = [area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus]

input_data = [7420,4,2,3,"yes","no","no","no","yes",2,"yes","furnished"]

Convert the string value in input_data to integer

In [6]:
schema = read_yaml(SCHEMA_FILE_PATH).COLUMNS

[2026-06-27 22:11:00,858: INFO: common: yaml file: yaml file\schema.yaml loaded successfully]


In [7]:
column_str_name = [k for k, v in schema.items() if v == "str"]

In [8]:
column_str_name

['mainroad',
 'guestroom',
 'basement',
 'hotwaterheating',
 'airconditioning',
 'prefarea',
 'furnishingstatus']

In [9]:
encoder = joblib.load(Path("artifacts/data_transformation/encoder.pkl"))

In [10]:
str_values = [x for x in input_data if isinstance(x, str)]

encoded_str_values = []

for value, col in zip(str_values, column_str_name):
    encoded_value = encoder[col].transform([value])[0]
    encoded_str_values.append(encoded_value)

print(encoded_str_values)

[1, 0, 0, 0, 1, 1, 0]


In [12]:
# retrieve the indices of string values in input_data
string_indices = [i for i, value in enumerate(input_data) if isinstance(value, str)]

# replace each string value in input_data with the corresponding value from the encoded_str_values
for idx, map_val in zip(string_indices, encoded_str_values):
    input_data[idx] = map_val
    
print("Original:", input_data)
print("Encoded: ", input_data)

Original: [7420, 4, 2, 3, 1, 0, 0, 0, 1, 2, 1, 0]
Encoded:  [7420, 4, 2, 3, 1, 0, 0, 0, 1, 2, 1, 0]


Normalization

In [13]:
df_transf_data = pd.read_csv(Path("artifacts/data_transformation/transf_data_file.csv"))
df_transf_data

,Unnamed: 0,mean_input_data,max_input_data,min_input_data
0,area,5154.144495,16200,1650
1,bedrooms,2.958716,6,1
2,bathrooms,1.266055,4,1
3,stories,1.782110,4,1
4,mainroad,0.857798,1,0
5,guestroom,0.178899,1,0
6,basement,0.357798,1,0
7,hotwaterheating,0.050459,1,0
8,airconditioning,0.307339,1,0
9,parking,0.685780,3,0


In [14]:
input_normalized_data = (input_data - df_transf_data ["mean_input_data"]) / (df_transf_data ["max_input_data"] - df_transf_data ["min_input_data"])

In [15]:
input_normalized_data

0     0.155729
1     0.208257
2     0.244648
3     0.405963
4     0.142202
5    -0.178899
6    -0.357798
7    -0.050459
8     0.692661
9     0.438073
10    0.766055
11   -0.529817
dtype: float64

Prediction

In [16]:
import numpy as np

In [ ]:
model = joblib.load(Path('artifacts/model_trainer/model.joblib'))

In [41]:
model.predict(np.array(input_normalized_data).reshape(1, 12))

array([6865222.0876264])

Update the entity

In [17]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PredictionNewDataConfig:
    model_path: Path
    transf_data_file: Path
    encoder_file: Path
    column_str_name: str

Update the configuration manager

In [18]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [19]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_prediction_new_data_config(self) -> PredictionNewDataConfig:
        config = self.config.prediction_new_data
        schema = self.schema.COLUMNS

        prediction_new_data_config = PredictionNewDataConfig(
            model_path = config.model_path,
            transf_data_file = config.transf_data_file,
            encoder_file = config.encoder_file,
            column_str_name = [k for k, v in schema.items() if v == "str"]
        )

        return prediction_new_data_config

Update the components

In [20]:
import joblib 
import pandas as pd
from pathlib import Path
import numpy as np

In [21]:

class PredictionNewDataPipeline:
    def __init__(self, config: PredictionNewDataConfig):
        self.config = config

        
    def Normalize_input_data(self, input_data):
        
        df_transf_data = pd.read_csv(self.config.transf_data_file)
        
        encoder =  joblib.load(self.config.encoder_file)
        
        str_values = [x for x in input_data if isinstance(x, str)]

        encoded_str_values = []

        for value, col in zip(str_values, self.config.column_str_name):
            encoded_value = encoder[col].transform([value])[0]
            encoded_str_values.append(encoded_value)

        # retrieve the indices of string values in input_data
        string_indices = [i for i, value in enumerate(input_data) if isinstance(value, str)]

        # replace each string value in input_data with the corresponding value from the encoded_str_values
        for idx, map_val in zip(string_indices, encoded_str_values):
            input_data[idx] = map_val

        input_normalized_data = (input_data - df_transf_data ["mean_input_data"]) / (df_transf_data ["max_input_data"] - df_transf_data ["min_input_data"])
        
        return input_normalized_data
    
    def predict(self, input_normalized_data):
        model = joblib.load(self.config.model_path)
        prediction = model.predict(np.array(input_normalized_data).reshape(1, 12))

        return prediction

Update the pipline

In [22]:
input_data_1 = [7420,4,2,3,"yes","no","no","no","yes",2,"yes","furnished"]

try:
 config = ConfigurationManager()   
 pred_new_data_config = config.get_prediction_new_data_config()
 pred_new_data = PredictionNewDataPipeline (config = pred_new_data_config)
 data_normalized_1 = pred_new_data.Normalize_input_data(input_data_1)
 pred_new_data.predict(data_normalized_1)
except Exception as e:
    raise e

[2026-06-27 22:11:51,392: INFO: common: yaml file: yaml file\config.yaml loaded successfully]
[2026-06-27 22:11:51,399: INFO: common: yaml file: yaml file\params.yaml loaded successfully]
[2026-06-27 22:11:51,408: INFO: common: yaml file: yaml file\schema.yaml loaded successfully]
[2026-06-27 22:11:51,415: INFO: common: created directory at: artifacts]


In [23]:
print(pred_new_data.predict(data_normalized_1))

[7990589.00452588]
